# 01. MoE 기초: total parameter와 active parameter

목표: Inkling 같은 sparse MoE 모델에서 total parameter와 active parameter가 왜 다른지 계산으로 이해합니다.

실행 방법: 위에서부터 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Inkling 제원 계산하기

공식 발표 기준 Inkling은 975B total parameters와 41B active parameters를 가집니다. Active 비율을 계산해 보면 token당 계산에 전체 capacity의 일부만 쓰는 MoE 구조를 직관적으로 볼 수 있습니다.

In [ ]:
inkling_total = 975_000_000_000
inkling_active = 41_000_000_000
inkling_small_total = 276_000_000_000
inkling_small_active = 12_000_000_000


def active_ratio(active, total):
    return active / total


for name, total, active in [
    ("Inkling", inkling_total, inkling_active),
    ("Inkling-Small", inkling_small_total, inkling_small_active),
]:
    print(f"{name}: active/total = {active_ratio(active, total):.2%}")

## 2. Toy router 만들기

실제 Inkling은 256 routed experts 중 6개와 2 shared experts를 사용합니다. 아래 toy router는 토큰 문자열을 해시처럼 점수화해 expert 6개를 고릅니다. 목적은 모델을 재현하는 것이 아니라 라우팅 개념을 익히는 것입니다.

In [ ]:
def score_token_for_expert(token, expert_id):
    # 재현 가능한 toy score입니다. 실제 router는 학습된 neural layer입니다.
    return sum(ord(ch) * (expert_id + 1) for ch in token) % 997


def route_token(token, expert_count=256, routed_active=6, shared_experts=2):
    scores = [(expert_id, score_token_for_expert(token, expert_id)) for expert_id in range(expert_count)]
    routed = [expert_id for expert_id, _score in sorted(scores, key=lambda item: item[1], reverse=True)[:routed_active]]
    shared = [f"shared_{i}" for i in range(shared_experts)]
    return routed + shared


tokens = ["audio", "image", "reasoning", "code", "forecast"]
for token in tokens:
    print(f"{token:10s} -> {route_token(token)}")

## 3. Dense model과 MoE model의 직관적 비교

Dense model은 모든 token이 거의 모든 parameter를 사용합니다. MoE는 token마다 일부 expert만 활성화합니다. 그래서 total capacity는 크지만 token당 계산량은 active parameter에 더 가깝습니다.

In [ ]:
def approximate_train_flops(parameters, tokens):
    # Dense Transformer 학습 비용을 대략 6NT로 보는 관례를 사용합니다.
    return 6 * parameters * tokens


def approximate_inference_compute(active_parameters, generated_tokens):
    # 실제 serving 비용은 attention, KV cache, batch, quantization 영향을 받습니다.
    # 여기서는 active parameter와 generated token이 비용에 큰 영향을 준다는 점만 보여 줍니다.
    return active_parameters * generated_tokens


generated_tokens = 10_000
dense_compute = approximate_inference_compute(inkling_total, generated_tokens)
moe_compute = approximate_inference_compute(inkling_active, generated_tokens)

print(f"dense-like compute units: {dense_compute:.2e}")
print(f"MoE active compute units: {moe_compute:.2e}")
print(f"relative reduction by active sparsity: {moe_compute / dense_compute:.2%}")

## 4. 하드웨어 요구량 해석

모델 카드 기준 BF16 checkpoint는 2TB 이상 aggregated VRAM, NVFP4 checkpoint는 600GB 이상 aggregated VRAM을 요구합니다. Open weights라고 해서 개인용 GPU 한 장에서 바로 실행 가능하다는 뜻은 아닙니다.

In [ ]:
hardware_options = [
    {"checkpoint": "BF16", "vram_gb": 2000, "example": "8x B300 or 16x H200"},
    {"checkpoint": "NVFP4", "vram_gb": 600, "example": "4x B300 W4A4 or 8x H200 W4A16"},
]

for option in hardware_options:
    print(f"{option['checkpoint']:5s} needs at least {option['vram_gb']} GB aggregated VRAM ({option['example']}).")